<a href="https://colab.research.google.com/github/jintubhuyan-2000/Spatial-Gradient-of-Highway-Induced-Land-Cover-Change/blob/main/SECTION_4_5_%E2%80%94_VEGETATION_AND_TREE_COVER_LOSSipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

# =============================================================================
# TEZPUR–NORTH LAKHIMPUR NATIONAL HIGHWAY CORRIDOR
# SECTION 4.5 — VEGETATION AND TREE-COVER LOSS
#
# COMPLETE GEOTIFF ANALYSIS
#
# IMPORTANT:
# The TreeCover Loss TIFF is OPTIONAL.
# If it does not exist, it will automatically be derived from:
#
#     TZPR_NLP_TreeCover_2016.tif
#     TZPR_NLP_TreeCover_2025.tif
#
# The derived file will be:
#
#     TZPR_NLP_TreeCover_Loss_2016_2025.tif
#
# Input directory:
#     /content/drive/MyDrive/TZPR_NLP_Research
#
# Output directory:
#     /content/drive/MyDrive/TZPR_NLP_Research/Vegetation_TreeCover_Analysis_4_5
#
# =============================================================================


import os
import math
import warnings

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from rasterio.transform import xy

warnings.filterwarnings("ignore")


# =============================================================================
# 1. USER SETTINGS
# =============================================================================

INPUT_DIR = "/content/drive/MyDrive/TZPR_NLP_Research"

OUTPUT_DIR = os.path.join(
    INPUT_DIR,
    "Vegetation_TreeCover_Analysis_4_5"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Number of raster rows processed at a time
CHUNK_ROWS = 256


# =============================================================================
# 2. FILE NAMES
# =============================================================================

FILES = {

    # -------------------------------------------------------------------------
    # Vegetation
    # -------------------------------------------------------------------------
    "vegetation_2016":
        "TZPR_NLP_Vegetation_2016.tif",

    "vegetation_2025":
        "TZPR_NLP_Vegetation_2025.tif",

    "vegetation_loss":
        "TZPR_NLP_Vegetation_Loss_2016_2025.tif",

    # -------------------------------------------------------------------------
    # Tree cover
    # -------------------------------------------------------------------------
    "tree_2016":
        "TZPR_NLP_TreeCover_2016.tif",

    "tree_2025":
        "TZPR_NLP_TreeCover_2025.tif",

    "tree_loss":
        "TZPR_NLP_TreeCover_Loss_2016_2025.tif",
}


# =============================================================================
# 3. FULL PATHS
# =============================================================================

PATHS = {
    key: os.path.join(INPUT_DIR, filename)
    for key, filename in FILES.items()
}


# =============================================================================
# 4. PRINT HEADER
# =============================================================================

print("=" * 80)
print("TEZPUR–NORTH LAKHIMPUR HIGHWAY CORRIDOR")
print("SECTION 4.5 — VEGETATION AND TREE-COVER LOSS")
print("=" * 80)

print()
print("Input directory:")
print(INPUT_DIR)

print()
print("Output directory:")
print(OUTPUT_DIR)

print()
print("Checking input files...")
print()


# =============================================================================
# 5. CHECK REQUIRED FILES
# =============================================================================

required_files = [
    "vegetation_2016",
    "vegetation_2025",
    "vegetation_loss",
    "tree_2016",
    "tree_2025",
]

missing_files = []

for key in required_files:

    path = PATHS[key]

    if os.path.exists(path):

        print(f"[OK]       {FILES[key]}")

    else:

        print(f"[MISSING]  {FILES[key]}")
        missing_files.append(FILES[key])


print()


if missing_files:

    print("Missing required files:")
    for f in missing_files:
        print(" -", f)

    raise FileNotFoundError(
        "Required input GeoTIFF files are missing. "
        "Please check the input directory."
    )


# =============================================================================
# 6. TREE-COVER LOSS FILE
# =============================================================================
#
# If the tree-cover loss TIFF already exists:
#     use it.
#
# If it does not exist:
#     derive it automatically from the 2016 and 2025 tree-cover masks.
#
# Definition:
#
# TreeCoverLoss =
#       TreeCover2016 == 1
#       AND
#       TreeCover2025 != 1
#
# Output:
#     1 = tree cover loss
#     0 = no tree cover loss
#
# =============================================================================


if os.path.exists(PATHS["tree_loss"]):

    print("[OK]       Existing TreeCover Loss TIFF found.")
    print("           Using existing:")
    print("           ", PATHS["tree_loss"])

else:

    print("[INFO]     TreeCover Loss TIFF not found.")
    print("[INFO]     Deriving TreeCover Loss from 2016 and 2025 rasters...")
    print()

    with rasterio.open(PATHS["tree_2016"]) as src16, \
         rasterio.open(PATHS["tree_2025"]) as src25:

        # ---------------------------------------------------------------------
        # Check raster dimensions
        # ---------------------------------------------------------------------

        if src16.width != src25.width or src16.height != src25.height:

            raise ValueError(
                "Tree-cover 2016 and 2025 rasters have different dimensions."
            )

        # ---------------------------------------------------------------------
        # Check CRS
        # ---------------------------------------------------------------------

        if src16.crs != src25.crs:

            raise ValueError(
                "Tree-cover 2016 and 2025 rasters have different CRS."
            )

        # ---------------------------------------------------------------------
        # Check transforms
        # ---------------------------------------------------------------------

        if not np.allclose(
            src16.transform,
            src25.transform
        ):

            raise ValueError(
                "Tree-cover 2016 and 2025 rasters have different spatial grids."
            )

        profile = src16.profile.copy()

        profile.update(
            dtype="uint8",
            count=1,
            compress="deflate",
            predictor=2,
            nodata=0
        )

        derived_loss_path = PATHS["tree_loss"]

        with rasterio.open(
            derived_loss_path,
            "w",
            **profile
        ) as dst:

            for row_start in range(
                0,
                src16.height,
                CHUNK_ROWS
            ):

                rows = min(
                    CHUNK_ROWS,
                    src16.height - row_start
                )

                window = Window(
                    col_off=0,
                    row_off=row_start,
                    width=src16.width,
                    height=rows
                )

                data16 = src16.read(
                    1,
                    window=window,
                    masked=True
                )

                data25 = src25.read(
                    1,
                    window=window,
                    masked=True
                )

                # -------------------------------------------------------------
                # Valid pixels
                # -------------------------------------------------------------

                valid16 = ~np.ma.getmaskarray(data16)
                valid25 = ~np.ma.getmaskarray(data25)

                valid = valid16 & valid25

                arr16 = np.asarray(
                    data16.filled(0)
                )

                arr25 = np.asarray(
                    data25.filled(0)
                )

                # -------------------------------------------------------------
                # Tree-cover loss
                #
                # 2016 tree = 1
                # 2025 tree != 1
                # -------------------------------------------------------------

                loss = (
                    (arr16 == 1)
                    &
                    (arr25 != 1)
                    &
                    valid
                )

                output = np.zeros(
                    arr16.shape,
                    dtype=np.uint8
                )

                output[loss] = 1

                dst.write(
                    output,
                    1,
                    window=window
                )

    print()
    print("[CREATED]  TreeCover Loss TIFF")
    print("           ", derived_loss_path)


# =============================================================================
# 7. RASTER INFORMATION FUNCTION
# =============================================================================

def raster_information(path):

    with rasterio.open(path) as src:

        return {
            "File": os.path.basename(path),
            "Width_pixels": src.width,
            "Height_pixels": src.height,
            "Bands": src.count,
            "CRS": str(src.crs),
            "PixelWidth": src.transform.a,
            "PixelHeight": abs(src.transform.e),
            "Bounds_Left": src.bounds.left,
            "Bounds_Right": src.bounds.right,
            "Bounds_Bottom": src.bounds.bottom,
            "Bounds_Top": src.bounds.top,
            "DataType": str(src.dtypes[0]),
            "NoData": src.nodata,
        }


# =============================================================================
# 8. PIXEL AREA FUNCTION
# =============================================================================
#
# Your rasters are in EPSG:4326.
#
# Therefore:
#
#     pixel width  = degrees longitude
#     pixel height = degrees latitude
#
# Pixel area cannot simply be calculated as:
#
#     pixel_width * pixel_height
#
# because those values are degrees, not metres.
#
# We calculate geographic pixel area using:
#
# A = R² × Δλ × [sin(phi2) - sin(phi1)]
#
# where:
#
#     R     = Earth radius
#     Δλ    = longitude width in radians
#     phi1  = southern latitude
#     phi2  = northern latitude
#
# =============================================================================


EARTH_RADIUS = 6378137.0


def calculate_row_pixel_area_km2(
    src,
    row
):

    transform = src.transform

    pixel_width_deg = abs(transform.a)
    pixel_height_deg = abs(transform.e)

    lon_width_rad = math.radians(
        pixel_width_deg
    )

    # Latitude at top and bottom of the row
    top_lat = transform.f + row * transform.e

    bottom_lat = (
        transform.f
        + (row + 1) * transform.e
    )

    lat1 = math.radians(
        min(top_lat, bottom_lat)
    )

    lat2 = math.radians(
        max(top_lat, bottom_lat)
    )

    area_m2 = (
        EARTH_RADIUS ** 2
        *
        lon_width_rad
        *
        (
            math.sin(lat2)
            -
            math.sin(lat1)
        )
    )

    return area_m2 / 1_000_000.0


# =============================================================================
# 9. BINARY RASTER STATISTICS
# =============================================================================

def calculate_binary_statistics(
    path,
    target_value=1,
    chunk_rows=CHUNK_ROWS
):

    total_valid_pixels = 0
    target_pixels = 0

    total_area_km2 = 0.0
    target_area_km2 = 0.0

    with rasterio.open(path) as src:

        for row_start in range(
            0,
            src.height,
            chunk_rows
        ):

            rows = min(
                chunk_rows,
                src.height - row_start
            )

            window = Window(
                0,
                row_start,
                src.width,
                rows
            )

            data = src.read(
                1,
                window=window,
                masked=True
            )

            mask = np.ma.getmaskarray(data)

            valid = ~mask

            arr = np.asarray(
                data.filled(0)
            )

            valid_count = int(
                np.count_nonzero(valid)
            )

            target_mask = (
                (arr == target_value)
                &
                valid
            )

            target_count = int(
                np.count_nonzero(target_mask)
            )

            total_valid_pixels += valid_count
            target_pixels += target_count

            # ---------------------------------------------------------------
            # Area for every row in this chunk
            # ---------------------------------------------------------------

            for local_row in range(rows):

                global_row = row_start + local_row

                pixel_area = (
                    calculate_row_pixel_area_km2(
                        src,
                        global_row
                    )
                )

                valid_row = valid[local_row]
                target_row = target_mask[local_row]

                total_area_km2 += (
                    np.count_nonzero(valid_row)
                    *
                    pixel_area
                )

                target_area_km2 += (
                    np.count_nonzero(target_row)
                    *
                    pixel_area
                )

    return {
        "valid_pixels": total_valid_pixels,
        "target_pixels": target_pixels,
        "total_area_km2": total_area_km2,
        "target_area_km2": target_area_km2,
        "target_area_ha": target_area_km2 * 100.0,
    }


# =============================================================================
# 10. ENDPOINT CHANGE ANALYSIS
# =============================================================================
#
# Calculates:
#
# 2016 persistence
# 2025 persistence
# loss
# gain
# unchanged non-target
#
# This is derived directly from the two endpoint masks.
#
# =============================================================================


def endpoint_change_analysis(
    path_2016,
    path_2025,
    target_value=1,
    chunk_rows=CHUNK_ROWS
):

    results = {
        "valid_pixels": 0,
        "target_2016_pixels": 0,
        "target_2025_pixels": 0,
        "loss_pixels": 0,
        "gain_pixels": 0,
        "persistent_pixels": 0,
    }

    areas = {
        "target_2016_km2": 0.0,
        "target_2025_km2": 0.0,
        "loss_km2": 0.0,
        "gain_km2": 0.0,
        "persistent_km2": 0.0,
    }

    with rasterio.open(path_2016) as src16, \
         rasterio.open(path_2025) as src25:

        if src16.width != src25.width:
            raise ValueError(
                "2016 and 2025 rasters have different widths."
            )

        if src16.height != src25.height:
            raise ValueError(
                "2016 and 2025 rasters have different heights."
            )

        if src16.crs != src25.crs:
            raise ValueError(
                "2016 and 2025 rasters have different CRS."
            )

        if not np.allclose(
            src16.transform,
            src25.transform
        ):
            raise ValueError(
                "2016 and 2025 rasters have different spatial grids."
            )

        for row_start in range(
            0,
            src16.height,
            chunk_rows
        ):

            rows = min(
                chunk_rows,
                src16.height - row_start
            )

            window = Window(
                0,
                row_start,
                src16.width,
                rows
            )

            data16 = src16.read(
                1,
                window=window,
                masked=True
            )

            data25 = src25.read(
                1,
                window=window,
                masked=True
            )

            mask16 = np.ma.getmaskarray(data16)
            mask25 = np.ma.getmaskarray(data25)

            valid = (
                (~mask16)
                &
                (~mask25)
            )

            arr16 = np.asarray(
                data16.filled(0)
            )

            arr25 = np.asarray(
                data25.filled(0)
            )

            target16 = (
                (arr16 == target_value)
                &
                valid
            )

            target25 = (
                (arr25 == target_value)
                &
                valid
            )

            loss = (
                target16
                &
                (~target25)
            )

            gain = (
                (~target16)
                &
                target25
            )

            persistence = (
                target16
                &
                target25
            )

            results["valid_pixels"] += int(
                np.count_nonzero(valid)
            )

            results["target_2016_pixels"] += int(
                np.count_nonzero(target16)
            )

            results["target_2025_pixels"] += int(
                np.count_nonzero(target25)
            )

            results["loss_pixels"] += int(
                np.count_nonzero(loss)
            )

            results["gain_pixels"] += int(
                np.count_nonzero(gain)
            )

            results["persistent_pixels"] += int(
                np.count_nonzero(persistence)
            )

            for local_row in range(rows):

                global_row = row_start + local_row

                pixel_area = (
                    calculate_row_pixel_area_km2(
                        src16,
                        global_row
                    )
                )

                areas["target_2016_km2"] += (
                    np.count_nonzero(
                        target16[local_row]
                    )
                    *
                    pixel_area
                )

                areas["target_2025_km2"] += (
                    np.count_nonzero(
                        target25[local_row]
                    )
                    *
                    pixel_area
                )

                areas["loss_km2"] += (
                    np.count_nonzero(
                        loss[local_row]
                    )
                    *
                    pixel_area
                )

                areas["gain_km2"] += (
                    np.count_nonzero(
                        gain[local_row]
                    )
                    *
                    pixel_area
                )

                areas["persistent_km2"] += (
                    np.count_nonzero(
                        persistence[local_row]
                    )
                    *
                    pixel_area
                )

    results.update(areas)

    results["target_2016_ha"] = (
        areas["target_2016_km2"] * 100
    )

    results["target_2025_ha"] = (
        areas["target_2025_km2"] * 100
    )

    results["loss_ha"] = (
        areas["loss_km2"] * 100
    )

    results["gain_ha"] = (
        areas["gain_km2"] * 100
    )

    results["persistent_ha"] = (
        areas["persistent_km2"] * 100
    )

    # Signed net change
    results["net_change_km2"] = (
        areas["target_2025_km2"]
        -
        areas["target_2016_km2"]
    )

    results["net_change_ha"] = (
        results["net_change_km2"]
        * 100
    )

    # Net loss percentage
    if areas["target_2016_km2"] > 0:

        results["loss_percent_of_2016"] = (
            areas["loss_km2"]
            /
            areas["target_2016_km2"]
            *
            100
        )

        results["net_change_percent"] = (
            results["net_change_km2"]
            /
            areas["target_2016_km2"]
            *
            100
        )

    else:

        results["loss_percent_of_2016"] = np.nan
        results["net_change_percent"] = np.nan

    return results


# =============================================================================
# 11. DERIVE ENDPOINT CHANGE RASTER
# =============================================================================

def create_change_raster(
    path_2016,
    path_2025,
    output_path,
    target_value=1,
    chunk_rows=CHUNK_ROWS
):

    print()
    print("Creating endpoint-derived change raster:")
    print("   ", output_path)

    with rasterio.open(path_2016) as src16, \
         rasterio.open(path_2025) as src25:

        if src16.width != src25.width:
            raise ValueError(
                "Raster widths do not match."
            )

        if src16.height != src25.height:
            raise ValueError(
                "Raster heights do not match."
            )

        if src16.crs != src25.crs:
            raise ValueError(
                "Raster CRS does not match."
            )

        if not np.allclose(
            src16.transform,
            src25.transform
        ):
            raise ValueError(
                "Raster transforms do not match."
            )

        profile = src16.profile.copy()

        profile.update(
            dtype="uint8",
            count=1,
            compress="deflate",
            predictor=2,
            nodata=0
        )

        with rasterio.open(
            output_path,
            "w",
            **profile
        ) as dst:

            for row_start in range(
                0,
                src16.height,
                chunk_rows
            ):

                rows = min(
                    chunk_rows,
                    src16.height - row_start
                )

                window = Window(
                    0,
                    row_start,
                    src16.width,
                    rows
                )

                data16 = src16.read(
                    1,
                    window=window,
                    masked=True
                )

                data25 = src25.read(
                    1,
                    window=window,
                    masked=True
                )

                valid16 = ~np.ma.getmaskarray(data16)
                valid25 = ~np.ma.getmaskarray(data25)

                valid = valid16 & valid25

                arr16 = np.asarray(
                    data16.filled(0)
                )

                arr25 = np.asarray(
                    data25.filled(0)
                )

                loss = (
                    (arr16 == target_value)
                    &
                    (arr25 != target_value)
                    &
                    valid
                )

                out = np.zeros(
                    arr16.shape,
                    dtype=np.uint8
                )

                out[loss] = 1

                dst.write(
                    out,
                    1,
                    window=window
                )

    print("[CREATED]")


# =============================================================================
# 12. RASTER INFORMATION CSV
# =============================================================================

print()
print("=" * 80)
print("RASTER INFORMATION")
print("=" * 80)

raster_info_rows = []

for key, path in PATHS.items():

    if os.path.exists(path):

        try:

            info = raster_information(path)
            info["Variable"] = key

            raster_info_rows.append(info)

            print(
                f"[OK] {FILES[key]} "
                f"{info['Width_pixels']} × "
                f"{info['Height_pixels']}"
            )

        except Exception as e:

            print(
                f"[WARNING] Could not inspect {FILES[key]}: {e}"
            )


raster_info_df = pd.DataFrame(
    raster_info_rows
)

raster_info_csv = os.path.join(
    OUTPUT_DIR,
    "01_Vegetation_TreeCover_Raster_Information.csv"
)

raster_info_df.to_csv(
    raster_info_csv,
    index=False
)


# =============================================================================
# 13. VEGETATION ENDPOINT ANALYSIS
# =============================================================================

print()
print("=" * 80)
print("VEGETATION ANALYSIS")
print("=" * 80)

vegetation_results = endpoint_change_analysis(
    PATHS["vegetation_2016"],
    PATHS["vegetation_2025"],
    target_value=1
)

print()
print(
    f"Vegetation 2016 : "
    f"{vegetation_results['target_2016_ha']:,.2f} ha"
)

print(
    f"Vegetation 2025 : "
    f"{vegetation_results['target_2025_ha']:,.2f} ha"
)

print(
    f"Vegetation loss  : "
    f"{vegetation_results['loss_ha']:,.2f} ha"
)

print(
    f"Vegetation gain  : "
    f"{vegetation_results['gain_ha']:,.2f} ha"
)

print(
    f"Net change       : "
    f"{vegetation_results['net_change_ha']:,.2f} ha"
)


# =============================================================================
# 14. TREE-COVER ENDPOINT ANALYSIS
# =============================================================================

print()
print("=" * 80)
print("TREE-COVER ANALYSIS")
print("=" * 80)

tree_results = endpoint_change_analysis(
    PATHS["tree_2016"],
    PATHS["tree_2025"],
    target_value=1
)

print()
print(
    f"Tree cover 2016 : "
    f"{tree_results['target_2016_ha']:,.2f} ha"
)

print(
    f"Tree cover 2025 : "
    f"{tree_results['target_2025_ha']:,.2f} ha"
)

print(
    f"Tree-cover loss  : "
    f"{tree_results['loss_ha']:,.2f} ha"
)

print(
    f"Tree-cover gain  : "
    f"{tree_results['gain_ha']:,.2f} ha"
)

print(
    f"Net change       : "
    f"{tree_results['net_change_ha']:,.2f} ha"
)


# =============================================================================
# 15. VERIFY / CREATE TREE LOSS RASTER
# =============================================================================

print()
print("=" * 80)
print("TREE-COVER LOSS RASTER")
print("=" * 80)

if not os.path.exists(PATHS["tree_loss"]):

    create_change_raster(
        PATHS["tree_2016"],
        PATHS["tree_2025"],
        PATHS["tree_loss"],
        target_value=1
    )

else:

    print(
        "[INFO] Existing tree-cover loss raster is being retained."
    )


# =============================================================================
# 16. VEGETATION LOSS RASTER STATISTICS
# =============================================================================

print()
print("=" * 80)
print("VEGETATION LOSS RASTER")
print("=" * 80)

vegetation_loss_stats = calculate_binary_statistics(
    PATHS["vegetation_loss"],
    target_value=1
)

print(
    f"Mapped vegetation-loss area: "
    f"{vegetation_loss_stats['target_area_ha']:,.2f} ha"
)


# =============================================================================
# 17. TREE-COVER LOSS RASTER STATISTICS
# =============================================================================

tree_loss_stats = calculate_binary_statistics(
    PATHS["tree_loss"],
    target_value=1
)

print(
    f"Mapped tree-cover-loss area: "
    f"{tree_loss_stats['target_area_ha']:,.2f} ha"
)


# =============================================================================
# 18. VEGETATION STATISTICS TABLE
# =============================================================================

vegetation_area_2016 = (
    vegetation_results["target_2016_ha"]
)

vegetation_area_2025 = (
    vegetation_results["target_2025_ha"]
)

vegetation_loss_ha = (
    vegetation_results["loss_ha"]
)

vegetation_gain_ha = (
    vegetation_results["gain_ha"]
)

vegetation_net_change_ha = (
    vegetation_results["net_change_ha"]
)

if vegetation_area_2016 > 0:

    vegetation_loss_percent = (
        vegetation_loss_ha
        /
        vegetation_area_2016
        *
        100
    )

    vegetation_remaining_percent = (
        vegetation_results["persistent_ha"]
        /
        vegetation_area_2016
        *
        100
    )

else:

    vegetation_loss_percent = np.nan
    vegetation_remaining_percent = np.nan


vegetation_stats_df = pd.DataFrame({

    "Indicator": [
        "Vegetation area 2016",
        "Vegetation area 2025",
        "Gross vegetation loss",
        "Gross vegetation gain",
        "Net vegetation change",
        "Vegetation loss (% of 2016)",
        "Persistent vegetation",
        "Persistent vegetation (% of 2016)",
        "Mapped vegetation-loss raster area",
        "Average annual gross vegetation loss",
        "Average annual net vegetation change",
    ],

    "Area_ha": [

        vegetation_area_2016,

        vegetation_area_2025,

        vegetation_loss_ha,

        vegetation_gain_ha,

        vegetation_net_change_ha,

        vegetation_loss_percent,

        vegetation_results["persistent_ha"],

        vegetation_remaining_percent,

        vegetation_loss_stats["target_area_ha"],

        vegetation_loss_ha / 9.0,

        vegetation_net_change_ha / 9.0,
    ],

    "Area_km2": [

        vegetation_area_2016 / 100,

        vegetation_area_2025 / 100,

        vegetation_loss_ha / 100,

        vegetation_gain_ha / 100,

        vegetation_net_change_ha / 100,

        np.nan,

        vegetation_results["persistent_km2"],

        np.nan,

        vegetation_loss_stats["target_area_km2"],

        vegetation_loss_ha / 100 / 9.0,

        vegetation_net_change_ha / 100 / 9.0,
    ]

})


vegetation_csv = os.path.join(
    OUTPUT_DIR,
    "03_Vegetation_Statistics.csv"
)

vegetation_stats_df.to_csv(
    vegetation_csv,
    index=False
)


# =============================================================================
# 19. TREE-COVER STATISTICS TABLE
# =============================================================================

tree_area_2016 = (
    tree_results["target_2016_ha"]
)

tree_area_2025 = (
    tree_results["target_2025_ha"]
)

tree_loss_ha = (
    tree_results["loss_ha"]
)

tree_gain_ha = (
    tree_results["gain_ha"]
)

tree_net_change_ha = (
    tree_results["net_change_ha"]
)

if tree_area_2016 > 0:

    tree_loss_percent = (
        tree_loss_ha
        /
        tree_area_2016
        *
        100
    )

    tree_remaining_percent = (
        tree_results["persistent_ha"]
        /
        tree_area_2016
        *
        100
    )

else:

    tree_loss_percent = np.nan
    tree_remaining_percent = np.nan


tree_stats_df = pd.DataFrame({

    "Indicator": [
        "Tree-cover area 2016",
        "Tree-cover area 2025",
        "Gross tree-cover loss",
        "Gross tree-cover gain",
        "Net tree-cover change",
        "Tree-cover loss (% of 2016)",
        "Persistent tree cover",
        "Persistent tree cover (% of 2016)",
        "Mapped tree-cover-loss raster area",
        "Average annual gross tree-cover loss",
        "Average annual net tree-cover change",
    ],

    "Area_ha": [

        tree_area_2016,

        tree_area_2025,

        tree_loss_ha,

        tree_gain_ha,

        tree_net_change_ha,

        tree_loss_percent,

        tree_results["persistent_ha"],

        tree_remaining_percent,

        tree_loss_stats["target_area_ha"],

        tree_loss_ha / 9.0,

        tree_net_change_ha / 9.0,
    ],

    "Area_km2": [

        tree_area_2016 / 100,

        tree_area_2025 / 100,

        tree_loss_ha / 100,

        tree_gain_ha / 100,

        tree_net_change_ha / 100,

        np.nan,

        tree_results["persistent_km2"],

        np.nan,

        tree_loss_stats["target_area_km2"],

        tree_loss_ha / 100 / 9.0,

        tree_net_change_ha / 100 / 9.0,
    ]

})


tree_csv = os.path.join(
    OUTPUT_DIR,
    "04_TreeCover_Statistics.csv"
)

tree_stats_df.to_csv(
    tree_csv,
    index=False
)


# =============================================================================
# 20. COMPLETE CHANGE STATISTICS
# =============================================================================

complete_rows = [

    {
        "Indicator": "Vegetation",
        "Area_2016_ha":
            vegetation_area_2016,
        "Area_2025_ha":
            vegetation_area_2025,
        "Gross_Loss_ha":
            vegetation_loss_ha,
        "Gross_Gain_ha":
            vegetation_gain_ha,
        "Net_Change_ha":
            vegetation_net_change_ha,
        "Loss_Percent_of_2016":
            vegetation_loss_percent,
        "Persistent_ha":
            vegetation_results["persistent_ha"],
        "Mapped_Loss_Raster_ha":
            vegetation_loss_stats["target_area_ha"],
    },

    {
        "Indicator": "Tree Cover",
        "Area_2016_ha":
            tree_area_2016,
        "Area_2025_ha":
            tree_area_2025,
        "Gross_Loss_ha":
            tree_loss_ha,
        "Gross_Gain_ha":
            tree_gain_ha,
        "Net_Change_ha":
            tree_net_change_ha,
        "Loss_Percent_of_2016":
            tree_loss_percent,
        "Persistent_ha":
            tree_results["persistent_ha"],
        "Mapped_Loss_Raster_ha":
            tree_loss_stats["target_area_ha"],
    }

]

complete_df = pd.DataFrame(
    complete_rows
)

complete_csv = os.path.join(
    OUTPUT_DIR,
    "06_Complete_Vegetation_TreeCover_Change_Statistics.csv"
)

complete_df.to_csv(
    complete_csv,
    index=False
)


# =============================================================================
# 21. RASTER AREA CHECK
# =============================================================================

raster_check_df = pd.DataFrame([

    {
        "Indicator":
            "Vegetation loss raster",
        "Raster_area_ha":
            vegetation_loss_stats["target_area_ha"],
        "Endpoint_loss_ha":
            vegetation_loss_ha,
        "Difference_ha":
            (
                vegetation_loss_stats["target_area_ha"]
                -
                vegetation_loss_ha
            ),
        "Raster_vs_Endpoint_percent":
            (
                vegetation_loss_stats["target_area_ha"]
                /
                vegetation_loss_ha
                *
                100
                if vegetation_loss_ha > 0
                else np.nan
            ),
    },

    {
        "Indicator":
            "Tree-cover loss raster",
        "Raster_area_ha":
            tree_loss_stats["target_area_ha"],
        "Endpoint_loss_ha":
            tree_loss_ha,
        "Difference_ha":
            (
                tree_loss_stats["target_area_ha"]
                -
                tree_loss_ha
            ),
        "Raster_vs_Endpoint_percent":
            (
                tree_loss_stats["target_area_ha"]
                /
                tree_loss_ha
                *
                100
                if tree_loss_ha > 0
                else np.nan
            ),
    }

])


raster_check_csv = os.path.join(
    OUTPUT_DIR,
    "05_Vegetation_TreeCover_Raster_Area_Check.csv"
)

raster_check_df.to_csv(
    raster_check_csv,
    index=False
)


# =============================================================================
# 22. MASTER MANUSCRIPT SUMMARY
# =============================================================================

master_summary = pd.DataFrame([

    {
        "Indicator":
            "Vegetation area in 2016",
        "Value":
            vegetation_area_2016,
        "Unit":
            "ha"
    },

    {
        "Indicator":
            "Vegetation area in 2025",
        "Value":
            vegetation_area_2025,
        "Unit":
            "ha"
    },

    {
        "Indicator":
            "Gross vegetation loss",
        "Value":
            vegetation_loss_ha,
        "Unit":
            "ha"
    },

    {
        "Indicator":
            "Gross vegetation gain",
        "Value":
            vegetation_gain_ha,
        "Unit":
            "ha"
    },

    {
        "Indicator":
            "Net vegetation change",
        "Value":
            vegetation_net_change_ha,
        "Unit":
            "ha"
    },

    {
        "Indicator":
            "Vegetation loss percentage",
        "Value":
            vegetation_loss_percent,
        "Unit":
            "%"
    },

    {
        "Indicator":
            "Mapped vegetation-loss raster",
        "Value":
            vegetation_loss_stats["target_area_ha"],
        "Unit":
            "ha"
    },

    {
        "Indicator":
            "Tree-cover area in 2016",
        "Value":
            tree_area_2016,
        "Unit":
            "ha"
    },

    {
        "Indicator":
            "Tree-cover area in 2025",
        "Value":
            tree_area_2025,
        "Unit":
            "ha"
    },

    {
        "Indicator":
            "Gross tree-cover loss",
        "Value":
            tree_loss_ha,
        "Unit":
            "ha"
    },

    {
        "Indicator":
            "Gross tree-cover gain",
        "Value":
            tree_gain_ha,
        "Unit":
            "ha"
    },

    {
        "Indicator":
            "Net tree-cover change",
        "Value":
            tree_net_change_ha,
        "Unit":
            "ha"
    },

    {
        "Indicator":
            "Tree-cover loss percentage",
        "Value":
            tree_loss_percent,
        "Unit":
            "%"
    },

    {
        "Indicator":
            "Mapped tree-cover-loss raster",
        "Value":
            tree_loss_stats["target_area_ha"],
        "Unit":
            "ha"
    },

    {
        "Indicator":
            "Vegetation loss per year",
        "Value":
            vegetation_loss_ha / 9.0,
        "Unit":
            "ha/year"
    },

    {
        "Indicator":
            "Tree-cover loss per year",
        "Value":
            tree_loss_ha / 9.0,
        "Unit":
            "ha/year"
    }

])


master_csv = os.path.join(
    OUTPUT_DIR,
    "02_Vegetation_TreeCover_Manuscript_Summary.csv"
)

master_summary.to_csv(
    master_csv,
    index=False
)


# =============================================================================
# 23. MANUSCRIPT-READY TEXT VALUES
# =============================================================================

txt_path = os.path.join(
    OUTPUT_DIR,
    "07_Vegetation_TreeCover_Manuscript_Values.txt"
)

with open(
    txt_path,
    "w",
    encoding="utf-8"
) as f:

    f.write("=" * 80 + "\n")
    f.write(
        "SECTION 4.5 — VEGETATION AND TREE-COVER LOSS\n"
    )
    f.write("=" * 80 + "\n\n")

    f.write("VEGETATION\n")
    f.write("-" * 80 + "\n")

    f.write(
        f"Vegetation area in 2016: "
        f"{vegetation_area_2016:,.2f} ha "
        f"({vegetation_area_2016/100:,.2f} km²)\n"
    )

    f.write(
        f"Vegetation area in 2025: "
        f"{vegetation_area_2025:,.2f} ha "
        f"({vegetation_area_2025/100:,.2f} km²)\n"
    )

    f.write(
        f"Gross vegetation loss: "
        f"{vegetation_loss_ha:,.2f} ha "
        f"({vegetation_loss_ha/100:,.2f} km²)\n"
    )

    f.write(
        f"Gross vegetation gain: "
        f"{vegetation_gain_ha:,.2f} ha "
        f"({vegetation_gain_ha/100:,.2f} km²)\n"
    )

    f.write(
        f"Net vegetation change: "
        f"{vegetation_net_change_ha:,.2f} ha "
        f"({vegetation_net_change_ha/100:,.2f} km²)\n"
    )

    f.write(
        f"Vegetation loss relative to 2016: "
        f"{vegetation_loss_percent:.2f}%\n"
    )

    f.write(
        f"Persistent vegetation: "
        f"{vegetation_results['persistent_ha']:,.2f} ha\n"
    )

    f.write(
        f"Mapped vegetation-loss raster: "
        f"{vegetation_loss_stats['target_area_ha']:,.2f} ha\n"
    )

    f.write("\n")
    f.write("TREE COVER\n")
    f.write("-" * 80 + "\n")

    f.write(
        f"Tree-cover area in 2016: "
        f"{tree_area_2016:,.2f} ha "
        f"({tree_area_2016/100:,.2f} km²)\n"
    )

    f.write(
        f"Tree-cover area in 2025: "
        f"{tree_area_2025:,.2f} ha "
        f"({tree_area_2025/100:,.2f} km²)\n"
    )

    f.write(
        f"Gross tree-cover loss: "
        f"{tree_loss_ha:,.2f} ha "
        f"({tree_loss_ha/100:,.2f} km²)\n"
    )

    f.write(
        f"Gross tree-cover gain: "
        f"{tree_gain_ha:,.2f} ha "
        f"({tree_gain_ha/100:,.2f} km²)\n"
    )

    f.write(
        f"Net tree-cover change: "
        f"{tree_net_change_ha:,.2f} ha "
        f"({tree_net_change_ha/100:,.2f} km²)\n"
    )

    f.write(
        f"Tree-cover loss relative to 2016: "
        f"{tree_loss_percent:.2f}%\n"
    )

    f.write(
        f"Persistent tree cover: "
        f"{tree_results['persistent_ha']:,.2f} ha\n"
    )

    f.write(
        f"Mapped tree-cover-loss raster: "
        f"{tree_loss_stats['target_area_ha']:,.2f} ha\n"
    )

    f.write("\n")
    f.write("IMPORTANT INTERPRETATION NOTE\n")
    f.write("-" * 80 + "\n")

    f.write(
        "Endpoint-derived loss represents pixels classified as the target "
        "cover in 2016 but not in 2025. The mapped loss raster may differ "
        "if it was generated using a different temporal or classification "
        "logic. Therefore, mapped-loss area and endpoint net change should "
        "not automatically be treated as identical quantities.\n"
    )

    f.write(
        "Vegetation and tree cover are reported as separate indicators and "
        "should not be interpreted as interchangeable classes unless their "
        "classification definitions are explicitly identical.\n"
    )

    f.write(
        "Observed vegetation or tree-cover loss within the corridor should "
        "not be described as being caused solely by the highway unless a "
        "spatial distance-gradient or other causal analysis supports that "
        "interpretation.\n"
    )


# =============================================================================
# 24. SAVE INDIVIDUAL ENDPOINT CHANGE TABLE
# =============================================================================

endpoint_change_df = pd.DataFrame([

    {
        "Indicator": "Vegetation",
        "2016_ha":
            vegetation_area_2016,
        "2025_ha":
            vegetation_area_2025,
        "Loss_ha":
            vegetation_loss_ha,
        "Gain_ha":
            vegetation_gain_ha,
        "Persistent_ha":
            vegetation_results["persistent_ha"],
        "Net_Change_ha":
            vegetation_net_change_ha,
        "Loss_percent":
            vegetation_loss_percent,
    },

    {
        "Indicator": "Tree Cover",
        "2016_ha":
            tree_area_2016,
        "2025_ha":
            tree_area_2025,
        "Loss_ha":
            tree_loss_ha,
        "Gain_ha":
            tree_gain_ha,
        "Persistent_ha":
            tree_results["persistent_ha"],
        "Net_Change_ha":
            tree_net_change_ha,
        "Loss_percent":
            tree_loss_percent,
    }

])


endpoint_csv = os.path.join(
    OUTPUT_DIR,
    "08_Endpoint_Vegetation_TreeCover_Change.csv"
)

endpoint_change_df.to_csv(
    endpoint_csv,
    index=False
)


# =============================================================================
# 25. FINAL OUTPUT LIST
# =============================================================================

print()
print("=" * 80)
print("ANALYSIS COMPLETED")
print("=" * 80)

print()
print("Output directory:")
print(OUTPUT_DIR)

print()
print("Generated files:")
print()

output_files = [

    "01_Vegetation_TreeCover_Raster_Information.csv",

    "02_Vegetation_TreeCover_Manuscript_Summary.csv",

    "03_Vegetation_Statistics.csv",

    "04_TreeCover_Statistics.csv",

    "05_Vegetation_TreeCover_Raster_Area_Check.csv",

    "06_Complete_Vegetation_TreeCover_Change_Statistics.csv",

    "07_Vegetation_TreeCover_Manuscript_Values.txt",

    "08_Endpoint_Vegetation_TreeCover_Change.csv",

    "TZPR_NLP_TreeCover_Loss_2016_2025.tif",

]

for filename in output_files:

    path = os.path.join(
        OUTPUT_DIR,
        filename
    )

    if os.path.exists(path):

        size_mb = (
            os.path.getsize(path)
            /
            (1024 * 1024)
        )

        print(
            f"[OK] {filename}"
            f"  ({size_mb:.2f} MB)"
        )

    else:

        print(
            f"[NOT FOUND] {filename}"
        )


# =============================================================================
# 26. FINAL NUMERICAL SUMMARY
# =============================================================================

print()
print("=" * 80)
print("FINAL NUMERICAL SUMMARY")
print("=" * 80)

print()
print("VEGETATION")
print(
    f"2016 area              : "
    f"{vegetation_area_2016:,.2f} ha"
)

print(
    f"2025 area              : "
    f"{vegetation_area_2025:,.2f} ha"
)

print(
    f"Gross loss             : "
    f"{vegetation_loss_ha:,.2f} ha"
)

print(
    f"Gross gain             : "
    f"{vegetation_gain_ha:,.2f} ha"
)

print(
    f"Net change             : "
    f"{vegetation_net_change_ha:,.2f} ha"
)

print(
    f"Loss percentage        : "
    f"{vegetation_loss_percent:.2f}%"
)

print(
    f"Mapped loss raster     : "
    f"{vegetation_loss_stats['target_area_ha']:,.2f} ha"
)


print()
print("TREE COVER")

print(
    f"2016 area              : "
    f"{tree_area_2016:,.2f} ha"
)

print(
    f"2025 area              : "
    f"{tree_area_2025:,.2f} ha"
)

print(
    f"Gross loss             : "
    f"{tree_loss_ha:,.2f} ha"
)

print(
    f"Gross gain             : "
    f"{tree_gain_ha:,.2f} ha"
)

print(
    f"Net change             : "
    f"{tree_net_change_ha:,.2f} ha"
)

print(
    f"Loss percentage        : "
    f"{tree_loss_percent:.2f}%"
)

print(
    f"Mapped loss raster     : "
    f"{tree_loss_stats['target_area_ha']:,.2f} ha"
)


print()
print("=" * 80)
print("SECTION 4.5 DATA EXTRACTION FINISHED")
print("=" * 80)



TEZPUR–NORTH LAKHIMPUR HIGHWAY CORRIDOR
SECTION 4.5 — VEGETATION AND TREE-COVER LOSS

Input directory:
/content/drive/MyDrive/TZPR_NLP_Research

Output directory:
/content/drive/MyDrive/TZPR_NLP_Research/Vegetation_TreeCover_Analysis_4_5

Checking input files...

[OK]       TZPR_NLP_Vegetation_2016.tif
[OK]       TZPR_NLP_Vegetation_2025.tif
[OK]       TZPR_NLP_Vegetation_Loss_2016_2025.tif
[OK]       TZPR_NLP_TreeCover_2016.tif
[OK]       TZPR_NLP_TreeCover_2025.tif

[INFO]     TreeCover Loss TIFF not found.
[INFO]     Deriving TreeCover Loss from 2016 and 2025 rasters...


[CREATED]  TreeCover Loss TIFF
            /content/drive/MyDrive/TZPR_NLP_Research/TZPR_NLP_TreeCover_Loss_2016_2025.tif

RASTER INFORMATION
[OK] TZPR_NLP_Vegetation_2016.tif 23084 × 9849
[OK] TZPR_NLP_Vegetation_2025.tif 23084 × 9849
[OK] TZPR_NLP_Vegetation_Loss_2016_2025.tif 23084 × 9849
[OK] TZPR_NLP_TreeCover_2016.tif 23084 × 9849
[OK] TZPR_NLP_TreeCover_2025.tif 23084 × 9849
[OK] TZPR_NLP_TreeCover_Loss_2016

In [1]:
from google.colab import drive
drive.mount('/content/drive')




Mounted at /content/drive
